<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP8.2A/MP8.2A_03_pinn.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP8.2A · Thermal Cycling with Natural Convection
## 03 · The Physics-Informed Network

*Mini project of the course **Deep Learning for Engineering**, Part 2. Five notebooks, run in order:*

00 The Problem → 01 The Data and the Physics → 02 The Reference Model and the Ground Truth → **03 The Physics-Informed Network** → 04 The Report

*Notebooks 00 to 02 are given and solved: read them, run them, and use what they build. Notebooks 03 and 04 are yours.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'heatsink.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08-heatsink-cooling/")
PROJECT = (['mp82a.py', 'MP8.2A_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP8.2A/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'heatsink', 'mp82a']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import heatsink as hs
import mp82a as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.keep_outputs("MP8.2A_outputs")                            # results kept between the notebooks (Google Drive on Colab)
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

## 1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else in this notebook is given, so that your month goes into the deep learning.

**The contract.** Your network is a PyTorch module called `net`, used as `net(X, Z, t, P)`:

| argument | meaning | shape |
|---|---|---|
| `X`, `Z` | $x/W$ and $z/t_b$, in 0 to 1 | `(n, 1)` |
| `t` | time in seconds, 0 to 600 | `(n, 1)` |
| `P` | the module's power in watts, 100 to 200 | `(n, 1)` |
| returns | the scaled rise $\hat\theta = \theta/\Theta$, with $\Theta$ = 46.57 K | `(n, 1)` |

Whatever you compute inside - features of $t$, built-in conditions, several sub-networks - stays inside `net`. The loss below and the evaluation at the end only call `net(X, Z, t, P)`.

## 2 · The loss, given

The function `losses(net, seed)` below draws fresh training points with the samplers of notebook 01 and returns the three residual terms of notebook 01's table: the heat equation inside, the module face, and the fin side. The ends and the start are not in it: build them into your network (notebook 01, section 2), or add their terms yourself. `balance_loss(net, seed)` is the energy balance at 64 random instants, computed with Gauss quadrature over the plate (notebook 01, section 2): optional, and in our experience decisive over a 600 s window.

Two details worth reading:

* **Every residual is divided by the Biot number**, $Bi = 0.0193$. All three are of that order here - the plate is thin and conducts well, so its temperature varies across the thickness by about $Bi\,\hat\theta$ - and left unscaled they would be a hundred times smaller than a network's usual error, and the optimiser would not see them.
* **The fin coefficient is evaluated at the network's own rise and held constant in the gradient** (`detach`). That is what the reference model's chord iterations do: the coefficient follows the solution, but the optimiser is not asked to differentiate through $\Delta T^{1/4}$, whose slope is infinite at $\Delta T = 0$.

In [ ]:
# The loss, given ---------------------------------------------------------------
def T(a):
    '''A NumPy array as an (n, 1) tensor on the device.'''
    return torch.tensor(np.asarray(a, dtype=float), device=DEVICE).reshape(-1, 1)

def d(u, x):
    '''du/dx by automatic differentiation, keeping the graph for a second derivative.'''
    return torch.autograd.grad(u.sum(), x, create_graph=True)[0]

TB = hs.T_BASE ** 2 / hs.ALPHA                           # s, the diffusion time through the thickness, 1.46 s

def losses(net, seed, n_in=1500, n_face=800):
    # inside the plate: the heat equation, t_b²/α ∂θ̂/∂t = ε² ∂²θ̂/∂X² + ∂²θ̂/∂Z²
    X, Z, t, P = map(T, mp.sample_interior(n_in, seed))
    for v in (X, Z, t):
        v.requires_grad_(True)                           # autograd differentiates with respect to these
    th = net(X, Z, t, P)
    r_pde = TB * d(th, t) - (hs.EPS ** 2 * d(d(th, X), X) + d(d(th, Z), Z))
    # the module face, Z = 1: ∂θ̂/∂Z = Bi g(X) (P/P0) s(t)
    Xt, tt, Pt = mp.sample_top(n_face, seed + 1)
    Zt = torch.ones(len(Xt), 1, device=DEVICE, requires_grad=True)
    top = net(T(Xt), Zt, T(tt), T(Pt))
    r_top = d(top, Zt) - hs.BI * T(hs.footprint(Xt) * mp.train_on(tt) * Pt / mp.P0)
    # the fin side, Z = 0: ∂θ̂/∂Z = h_eff(θ) t_b / k · θ̂, the coefficient at the network's own rise
    Xb, tb, Pb = mp.sample_bottom(n_face, seed + 2)
    Zb = torch.zeros(len(Xb), 1, device=DEVICE, requires_grad=True)
    bot = net(T(Xb), Zb, T(tb), T(Pb))
    bi = T(mp.h_eff(bot.detach().cpu().numpy().ravel() * hs.THETA) * hs.T_BASE / hs.K_AL)
    r_fin = d(bot, Zb) - bi * bot
    return {"pde": ((r_pde / hs.BI) ** 2).mean(),     # each residual divided by Bi: all three are of its order
            "module": ((r_top / hs.BI) ** 2).mean(),
            "fins": ((r_fin / hs.BI) ** 2).mean()}

# the energy balance at random instants (optional; notebook 01, section 2):
#   t_b²/α d<θ̂>/dt = Bi (P/P0) s(t) - <h_eff t_b/k θ̂> at Z = 0
_gx, _wx = np.polynomial.legendre.leggauss(32)           # Gauss points across the base ...
_gz, _wz = np.polynomial.legendre.leggauss(8)            # ... and through the thickness
GX, WX = T((_gx + 1) / 2).ravel(), T(_wx / 2).ravel()    # on (0, 1); the weights sum to one, so a sum is a mean
GZ, WZ = T((_gz + 1) / 2).ravel(), T(_wz / 2).ravel()

def balance_loss(net, seed, n_t=64):
    rng = np.random.default_rng(seed)
    tq = T(mp.sample_times(n_t, rng, 0.25)).requires_grad_(True)       # instants
    Pq = T(rng.uniform(mp.P_MIN, mp.P_MAX, n_t))                       # and powers
    nx, nz = len(GX), len(GZ)
    X = GX.repeat_interleave(nz).repeat(n_t).reshape(-1, 1)            # every quadrature point at every instant
    Z = GZ.repeat(nx).repeat(n_t).reshape(-1, 1)
    w = (WX[:, None] * WZ[None, :]).reshape(-1).repeat(n_t).reshape(-1, 1)
    th = net(X, Z, tq.repeat_interleave(nx * nz, 0), Pq.repeat_interleave(nx * nz, 0))
    mean = (w * th).reshape(n_t, -1).sum(1, keepdim=True)              # <θ̂> over the plate at each instant
    rate = d(mean, tq)                                                 # its time derivative, by autograd
    Xf = GX.repeat(n_t).reshape(-1, 1)
    thf = net(Xf, torch.zeros_like(Xf), tq.repeat_interleave(nx, 0), Pq.repeat_interleave(nx, 0))
    bi = T(mp.h_eff(thf.detach().cpu().numpy().ravel() * hs.THETA) * hs.T_BASE / hs.K_AL)
    out = (WX.repeat(n_t).reshape(-1, 1) * bi * thf).reshape(n_t, -1).sum(1, keepdim=True)
    s = T(mp.train_on(tq.detach().cpu().numpy().ravel()))
    r = TB * rate - (hs.BI * s * Pq / mp.P0 - out)
    return ((r / hs.BI) ** 2).mean()

## 3 · TODO 1 · Your network

Write the class `Net` and make `net = Net().to(DEVICE)`. Decide:

* **what the network sees.** The four inputs, scaled to order one; or features built from them - functions of $X$ whose slope vanishes at the ends, functions of $t$ that carry the duty cycle's switching (notebook 01, section 2);
* **what is built in.** The insulated ends, the start at the air's temperature, the scaling of the rise with the power;
* **its size.** Layers, neurons, activation. Remember why Part 2 uses tanh, not ReLU.

Two things the problem tells you, from notebooks 00 and 01. The history has twenty kinks, one at each switch. And the module's flux steps from zero to 50 kW/m² at the footprint's edges: across the base the field needs more detail than one smooth function of $X$ gives.

In [ ]:
# TODO 1 - your network. net(X, Z, t, P) -> θ̂, every argument an (n, 1) tensor (see the table above)
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your layers and any trainable numbers

    def forward(self, X, Z, t, P):
        ...                                              # return θ̂, shape (n, 1)

torch.manual_seed(88)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## 4 · TODO 2 · Your training

Train `net` with `losses` (and, if you choose, `balance_loss`). Decide the optimiser or optimisers, the number of steps, the learning rate and its schedule, the weight of each term, how many points and whether they are drawn afresh each step. Keep the history of each term for TODO 3, and **measure the training time**: set `TRAIN_SECONDS`, which the evaluation reports as the offline cost.

The course's recipe (Adam, then L-BFGS) is a good start. Check the score as you go: `quick_score()` below gives the worst peak or trough error at the three test powers in a second, from the ground truth.

In [ ]:
# A quick score, given: the worst peak or trough error (K) at the test powers
G = np.load("MP8.2A_truth.npz")
def quick_score():
    tq = np.arange(0.0, mp.T_END + 1e-9, 0.5); worst = 0.0
    for P in mp.P_TEST:
        with torch.no_grad():
            th = net(T(np.full(tq.size, 0.5)), T(np.ones(tq.size)), T(tq), T(np.full(tq.size, P))).cpu().numpy().ravel() * hs.THETA
        p, q = mp.cycle_extremes(tq, th)
        worst = max(worst, np.abs(p - G[f"peaks_{int(P)}W"]).max(), np.abs(q - G[f"troughs_{int(P)}W"]).max())
    return worst

In [ ]:
# TODO 2 - your training
history = {"pde": [], "module": [], "fins": [], "balance": []}
t0 = time.perf_counter()
...                                                      # your optimiser(s) and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst peak or trough error {quick_score():.3f} K")

## 5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the loss terms during training, and the centre's history at one test power against the ground truth, with its error. Save each figure you want in the report as `MP8.2A_report_<name>.png` (in `cc.output_path`): notebook 04 embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## 6 · The evaluation, given

This cell asks your trained `net` the questions notebook 04 scores, and saves the answers to `MP8.2A_results.npz` (kept in your Google Drive, or downloaded):

* the centre's rise every 0.5 s at the test powers 115, 165 and 195 W, at 90 and 210 W outside the range, and at eleven powers from 100 to 200 W;
* the field at 600 s at 165 W on the ground truth's grid;
* the energy balance of your network at 165 W: heat in, heat through the fins, heat stored at 600 s;
* the **online time** of a new power on this runtime - the ten peaks and troughs at 165 W, from one call of `net` - and the same for 10 and 100 powers at once, against the reference model run 1, 10 and 100 times on the same runtime.

Set `TRAIN_SECONDS` to the offline training time you measured; it is reported beside the online time.

In [ ]:
# The evaluation, given ------------------------------------------------------------
import json
net.eval()
tq = np.arange(0.0, mp.T_END + 1e-9, 0.5)                # s, the instants of the history
def centre_history(powers):
    # the centre's rise (K) at every instant, for several powers in one call of the network
    powers = np.atleast_1d(powers)
    n = len(tq)
    X = torch.full((n * len(powers), 1), 0.5, device=DEVICE)
    Z = torch.ones_like(X)
    t = T(np.tile(tq, len(powers)))
    P = T(np.repeat(powers, n))
    with torch.no_grad():
        return (net(X, Z, t, P) * hs.THETA).reshape(len(powers), n).cpu().numpy()

def extremes(hist):
    return np.array([mp.cycle_extremes(tq, h) for h in hist])     # (powers, 2, cycles)

powers = {"test": np.array(mp.P_TEST), "outside": np.array([90.0, 210.0]), "sweep": np.linspace(100, 200, 11)}
out = {f"centre_{k}": centre_history(v) for k, v in powers.items()}
out.update({f"powers_{k}": v for k, v in powers.items()}, t=tq)

# the field at 600 s at 165 W, on the ground truth's grid
G = np.load("MP8.2A_truth.npz")
XX, ZZ = np.meshgrid(G["x"] / hs.W_BASE, G["z"] / hs.T_BASE)
with torch.no_grad():
    out["field_165W"] = (net(T(XX), T(ZZ), T(np.full(XX.size, mp.T_END)), T(np.full(XX.size, 165.0)))
                         * hs.THETA).reshape(XX.shape).cpu().numpy()

# the network's energy balance at 165 W
xs = np.linspace(0, 1, 201); ts = np.arange(0.25, mp.T_END, 0.5)
XB, TT = np.meshgrid(xs, ts)
with torch.no_grad():
    fin_side = (net(T(XB), T(np.zeros(XB.size)), T(TT), T(np.full(XB.size, 165.0))) * hs.THETA).cpu().numpy().reshape(XB.shape)
    end_field = (net(T(XX), T(ZZ), T(np.full(XX.size, mp.T_END)), T(np.full(XX.size, 165.0))) * hs.THETA).cpu().numpy()
q_fins = hs.DEPTH * np.trapezoid(mp.h_eff(fin_side) * fin_side, xs * hs.W_BASE, axis=1)    # W through the fins
heat_out = float(np.sum(q_fins) * 0.5)
stored = float(hs.RHO_CP * hs.DEPTH * np.trapezoid(np.trapezoid(end_field.reshape(XX.shape), G["x"], axis=1), G["z"]))
out["balance_165W"] = np.array([165.0 * mp.T_ON * mp.CYCLES, heat_out, stored])

# online time: the network against the reference model, on this runtime
timing = {}
for k in (1, 10, 100):
    ps = np.linspace(101.0, 199.0, k)
    timing[f"net_{k}"] = middle(lambda: extremes(centre_history(ps)))
    timing[f"ref_{k}"] = middle(lambda: [mp.reference(p) for p in ps], reps=3 if k == 100 else 5)
out.update({k: np.array(v) for k, v in timing.items()})
out["train_seconds"] = np.array(TRAIN_SECONDS)
out["n_weights"] = np.array(sum(p.numel() for p in net.parameters()))
out["device"] = np.array(DEVICE)
path = cc.output_path("MP8.2A_results.npz")
np.savez_compressed(path, **out)
torch.save(net.state_dict(), cc.output_path("MP8.2A_net.pt"))
cc.saved(path, cc.output_path("MP8.2A_net.pt"))
print(f"online, one new power: network {timing['net_1']*1e3:.2f} ms, reference {timing['ref_1']*1e3:.1f} ms "
      f"({timing['ref_1']/timing['net_1']:.0f} times)")
print(f"energy at 165 W: in {out['balance_165W'][0]:.0f} J, fins {heat_out:.0f} J, stored {stored:.0f} J")